In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
!pip install -U torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.8 MB/s eta 0:00:00


In [2]:
"""
chat.py
-------
File dùng lúc CHẠY THẬT (inference) - sau khi đã train xong (có output_model/).

Luồng:
  chunks (từ vector DB) + question
        -> build_prompt() (từ prompt_template.py)
        -> model đã train (base model + adapter LoRA)
        -> câu trả lời

Cách chạy thử (demo, chưa nối vector DB thật):
    python chat.py
"""

'\nchat.py\n-------\nFile dùng lúc CHẠY THẬT (inference) - sau khi đã train xong (có output_model/).\n\nLuồng:\n  chunks (từ vector DB) + question\n        -> build_prompt() (từ prompt_template.py)\n        -> model đã train (base model + adapter LoRA)\n        -> câu trả lời\n\nCách chạy thử (demo, chưa nối vector DB thật):\n    python chat.py\n'

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

import sys
sys.path.append("/content/drive/MyDrive/MockProject_NguyenTuanPhat/Day 12")
from prompt_template import build_prompt

In [12]:
# ============================================================
# CẤU HÌNH - chỉnh đúng với train.py đã dùng
# ============================================================

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # phải giống MODEL_NAME trong train.py
ADAPTER_DIR = "/content/drive/MyDrive/MockProject_NguyenTuanPhat/Day 12/output_model" # thư mục chứa adapter đã train

USE_GPU = torch.cuda.is_available()

In [6]:
# ============================================================
# LOAD MODEL (base + adapter) - chỉ cần load 1 LẦN lúc khởi động chatbot,
# không load lại mỗi lần user hỏi (rất tốn thời gian nếu load lại)
# ============================================================

def load_chat_model():
    print("Đang load model...")

    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

    if USE_GPU:
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            dtype=torch.float16,
            device_map="auto",
        )
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            dtype=torch.float32,
            device_map={"": "cpu"},
        )

    # Gắn adapter (kiến thức đã train) vào model gốc
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    model.eval()

    print("Load model xong.")
    return model, tokenizer


In [7]:
# ============================================================
# SINH CÂU TRẢ LỜI
# ============================================================

def generate_answer(model, tokenizer, chunks: list[str], question: str, max_new_tokens: int = 300) -> str:
    """
    Args:
        chunks: danh sách đoạn văn bản liên quan (do vector DB trả về)
        question: câu hỏi của user

    Returns:
        Câu trả lời dạng string
    """
    messages = build_prompt(chunks, question)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Chỉ lấy phần token MỚI được sinh ra (bỏ phần prompt input đi)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()

In [8]:
# ============================================================
# DEMO CHẠY THỬ (giả lập chunks, vì chưa nối vector DB thật)
# ============================================================

def main():
    model, tokenizer = load_chat_model()

    # Đây là chunks GIẢ LẬP -- lúc chạy thật, phần này sẽ do vector DB
    # của đồng nghiệp bạn trả về, không phải viết tay như thế này.
    demo_chunks = [
        "Signs and symptoms of adult ALL include fever, feeling tired, "
        "and easy bruising or bleeding. Check with your doctor if you have "
        "weakness, night sweats, easy bruising, petechiae, shortness of breath, "
        "weight loss, bone or stomach pain, or painless lumps."
    ]
    demo_question = "Triệu chứng của bệnh bạch cầu cấp ở người lớn là gì?"

    print("\n" + "=" * 50)
    print("CÂU HỎI:", demo_question)
    print("=" * 50)

    answer = generate_answer(model, tokenizer, demo_chunks, demo_question)

    print("TRẢ LỜI:", answer)
    print("=" * 50)

In [15]:
if __name__ == "__main__":
    main()

Đang load model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Load model xong.

CÂU HỎI: Triệu chứng của bệnh bạch cầu cấp ở người lớn là gì?
TRẢ LỜI: Triệu chứng của bệnh bạch cầu cấp ở người lớn là triệu chứng của các tình trạng bạch cầu cấp (thường gọi là bạch cầu thiếu) hoặc bạch cầu thay thế. Điều này xảy ra khi cơ thể không có đủ bạch cầu, làm giảm khả năng chuyển hóa và sản xuất máu. 

Nó có thể dẫn đến một loạt các triệu chứng khác nhau tùy thuộc vào tình trạng bạch cầu. Các triệu chứng phổ biến bao gồm:

- Sưng chân, cổ, tai, hoặc gáy
- Đau bụng hoặc đau họng
- Đau não
- Tiredness, khó ngủ, suy giảm sức khỏe
- Nước tiểu mạn tính 
- Bụng không đều, thường là trong mùa thai nghén

Hơn nữa, nếu bạch cầu thiếu hoặc bạch cầu thay thế quá mức, bạn cũng có thể gặp các triệu chứng sau:

- Sưng tím trên cơ thể
- Đau cơ thể
- Đau đầu
- Rối loạn cảm giác
- Cough
- Cảm thấy ấm hoặc lạnh hơn bình thường
- Cực kỳ mệt mỏi hoặc mất sức
- Thirst
- Nắng mắt
- Đôi mắt mờ nhợt
- Nỗi đau âm đạo
- Mất khả năng di chuyển vùng da bị bỏng
- Đau sống lưng hoặc đa